# 03 Preprocessing — WLC-Eignungsfaktoren

**PA1 ZHAW IUNR** | Bächler, Haag, Reichlin | Betreuer: Patrick Laube

Prüft und verarbeitet alle Datensätze für die Weighted Linear Combination.

**Einziger Input aus vorherigen Notebooks:**
```
data/processed/constraints/constraint_mask_s2.tif
```
Dieses Notebook ist unabhängig von 01/02a ausführbar — eine beliebige binäre Maske (1 = geeignet, 0 = ausgeschlossen) reicht als Input.

**Output:** 10 normalisierte Raster + Akzeptanz-Layer + Distanz-Raster

## 1. Setup

In [32]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
import pickle
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from rasterio.warp import reproject
from scipy.ndimage import distance_transform_edt
import fiona
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

RAW = Path("../data/raw")
PROC = Path("../data/processed")
OUT = Path("../outputs/figures")
for d in [PROC / "criteria", PROC / "distances", OUT]:
    d.mkdir(parents=True, exist_ok=True)

CRS = "EPSG:2056"
RES = 25
NODATA = -9999.0

## 2. Constraint-Maske & Referenzraster laden

In [33]:
mask_path = PROC / "constraints/constraint_mask_s2.tif"
with rasterio.open(mask_path) as src:
    constraint_mask = src.read(1)
    transform = src.transform
    ref_profile = src.profile.copy()
    ref_profile.update(dtype="float32", nodata=NODATA)
    height, width = constraint_mask.shape

# DEM (aus 02a)
with rasterio.open(PROC / "dem/dem_gr_25m.tif") as src:
    dem = src.read(1)
with rasterio.open(PROC / "dem/slope_gr_25m.tif") as src:
    slope = src.read(1)
with rasterio.open(PROC / "dem/aspect_gr_25m.tif") as src:
    aspect = src.read(1)

valid = (dem != NODATA) & (constraint_mask == 1)
print(f"Constraint-Maske: {constraint_mask.sum():,} Pixel ({constraint_mask.sum()*RES*RES/1e6:.0f} km²)")
print(f"Raster: {width}×{height} @ {RES}m")

Constraint-Maske: 876,268 Pixel (548 km²)
Raster: 5641×4010 @ 25m


In [34]:
# Hilfsfunktionen

def normalize_linear(data, low, high, invert=False):
    out = np.full_like(data, NODATA)
    m = valid & (data != NODATA)
    vals = np.clip((data[m] - low) / (high - low), 0, 1)
    out[m] = 1 - vals if invert else vals
    return out

def normalize_piecewise(data, breakpoints, values):
    out = np.full_like(data, NODATA)
    m = valid & (data != NODATA)
    out[m] = np.interp(data[m], breakpoints, values)
    return out

def save_criterion(data, name):
    path = PROC / f"criteria/{name}.tif"
    with rasterio.open(path, "w", **ref_profile) as dst:
        dst.write(data, 1)
    v = data[valid & (data != NODATA)]
    print(f"  {name}: {v.min():.3f}–{v.max():.3f} (mean {v.mean():.3f})")

def rasterize_to_grid(gdf, burn=1):
    shapes = [(g, burn) for g in gdf.to_crs(CRS).geometry if g is not None]
    if not shapes:
        return np.zeros((height, width), dtype=np.uint8)
    return rasterize(shapes, out_shape=(height, width), transform=transform, fill=0, dtype=np.uint8)

def compute_distance(binary_raster, name):
    dist = distance_transform_edt((binary_raster == 0).astype(float), sampling=(RES, RES)).astype(np.float32)
    with rasterio.open(PROC / f"distances/{name}.tif", "w", **ref_profile) as dst:
        dst.write(dist, 1)
    print(f"  {name}: max {dist[valid].max():.0f} m")
    return dist

## 3. Datenprüfung — WLC-Datensätze

Qualitätskontrolle der Datensätze die NUR für WLC relevant sind (nicht für Constraints).

In [35]:
print("=== WLC-Datensätze prüfen ===\n")

# BFE Solar
for deg in ["0", "30", "75", "90"]:
    parent = RAW / "solar" / f"solarenergie-einstrahlung_{deg}_grad_2056.gpkg"
    gpkg = next(parent.glob("*.gpkg"), None) if parent.is_dir() else (parent if parent.exists() else None)
    if gpkg:
        gdf = gpd.read_file(gpkg, rows=slice(0, 1))
        print(f"BFE Solar {deg}°: ✓ CRS={gdf.crs}")
    else:
        print(f"BFE Solar {deg}°: ⚠ FEHLT")

# SPASS Schnee
spass = sorted((RAW / "snow").glob("**/*.nc"))
print(f"SPASS Schnee: {'✓ ' + spass[0].name if spass else '⚠ FEHLT'}")

# swissTLM3D (Strassen, Leitungen, Staumauern — WLC-spezifisch)
tlm_path = RAW / "tlm3d/swisstlm3d_2026-02-24_2056_5728.gpkg"
if tlm_path.is_dir():
    tlm_path = next(tlm_path.glob("*.gpkg"))
print(f"swissTLM3D: {'✓' if tlm_path.exists() else '⚠ FEHLT'}")

=== WLC-Datensätze prüfen ===

BFE Solar 0°: ✓ CRS=EPSG:2056
BFE Solar 30°: ✓ CRS=EPSG:2056
BFE Solar 75°: ✓ CRS=EPSG:2056
BFE Solar 90°: ✓ CRS=EPSG:2056
SPASS Schnee: ✓ SWECLQMD_ch01h.swiss.lv95_WY_1962_2023.nc
swissTLM3D: ✓


## 4. BFE Solar → Raster

In [ ]:
import time

bbox = (transform.c, transform.f + height * transform.e,
        transform.c + width * transform.a, transform.f)

# ── BFE Solar: alle 4 Neigungen einlesen, dann auf 45° interpolieren ──
# Verfügbare Neigungswinkel: 0°, 30°, 75°, 90°
# Optimaler Installationswinkel: 45° → per np.interp pixelweise interpoliert

TILT_ANGLES = [30, 75]
COL_JAHR  = "Globalstrahlung_Jahressumme_kWhm2"
COL_WINTER = "Globalstrahlung_Wintersumme_kWhm2"

# Schritt 1: Alle 4 Datensätze einlesen & rasterisieren
solar_jahr = {}    # {tilt_deg: raster}
solar_winter = {}  # {tilt_deg: raster}

for deg in TILT_ANGLES:
    t0 = time.time()
    parent = RAW / "solar" / f"solarenergie-einstrahlung_{deg}_grad_2056.gpkg"
    gpkg = next(parent.glob("*.gpkg")) if parent.is_dir() else parent
    if not gpkg.exists():
        print(f"⚠ BFE Solar {deg}°: FEHLT — übersprungen")
        continue
    print(f"Lese BFE Solar {deg}° …")
    gdf_s = gpd.read_file(gpkg, bbox=bbox)
    print(f"  {len(gdf_s)} Features | Jahr: {gdf_s[COL_JAHR].min():.0f}–{gdf_s[COL_JAHR].max():.0f} | Winter: {gdf_s[COL_WINTER].min():.0f}–{gdf_s[COL_WINTER].max():.0f} kWh/m²")

    for col, store in [(COL_JAHR, solar_jahr), (COL_WINTER, solar_winter)]:
        shapes = [(g, v) for g, v in zip(gdf_s.geometry, gdf_s[col]) if g is not None and not np.isnan(v)]
        store[deg] = rasterize(shapes, out_shape=(height, width), transform=transform, fill=NODATA, dtype=np.float32)

    print(f"  ✓ {time.time()-t0:.0f}s")
    del gdf_s
    
print(f"\nGeladene Neigungen: {sorted(solar_jahr.keys())}")

# Schritt 2: Pixelweise Interpolation auf 45°
TARGET_TILT = 45
available_tilts = sorted(solar_jahr.keys())

def interpolate_tilt(raster_dict, tilts, target):
    """Pixelweise lineare Interpolation über Neigungswinkel (vektorisiert)."""
    stack = np.stack([raster_dict[t] for t in tilts], axis=0)  # (n_tilts, H, W)
    out = np.full((height, width), NODATA, dtype=np.float32)
    m = np.all(stack != NODATA, axis=0)  # nur wo alle Neigungen gültig
    tilts_arr = np.array(tilts, dtype=np.float64)
    idx = np.searchsorted(tilts_arr, target, side="right")
    idx = np.clip(idx, 1, len(tilts) - 1)
    t_lo, t_hi = tilts_arr[idx - 1], tilts_arr[idx]
    w = (target - t_lo) / (t_hi - t_lo) if t_hi != t_lo else 0.0
    out[m] = (1 - w) * stack[idx - 1][m] + w * stack[idx][m]
    return out

print(f"\nInterpoliere auf {TARGET_TILT}° (aus {available_tilts}) …")
t0 = time.time()
rast_jahr_45  = interpolate_tilt(solar_jahr,  available_tilts, TARGET_TILT)
rast_winter_45 = interpolate_tilt(solar_winter, available_tilts, TARGET_TILT)
print(f"  ✓ {time.time()-t0:.0f}s")

# Schritt 3: Normalisieren & speichern
# F01: Globalstrahlung — Linear [0,1]: (x - min) / (max - min), höher = besser
#      Ref: Kahl et al. (2019) PNAS, Frischholz et al. (2024)
v_j = rast_jahr_45[valid & (rast_jahr_45 != NODATA)]
f01 = normalize_linear(rast_jahr_45, v_j.min(), v_j.max())
save_criterion(f01, "f01_globalstrahlung")
print(f"  F01: Jahressumme bei {TARGET_TILT}° | {v_j.min():.0f}–{v_j.max():.0f} kWh/m²/a")

# F02: Wintereinstrahlung Okt–Mär — Linear [0,1]: (x - min) / (max - min), höher = besser
#      Ref: Art. 71a EnG (500 kWh/kW Winterstrom), Frischholz et al. (2024)
v_w = rast_winter_45[valid & (rast_winter_45 != NODATA)]
f02 = normalize_linear(rast_winter_45, v_w.min(), v_w.max())
save_criterion(f02, "f02_wintereinstrahlung")
print(f"  F02: Wintersumme bei {TARGET_TILT}° | {v_w.min():.0f}–{v_w.max():.0f} kWh/m²")

Lese BFE Solar 0° …
  4150565 Features | Jahr: 0–1611 | Winter: 0–551 kWh/m²


KeyboardInterrupt: 

In [ ]:
print(f"F01: Globalstrahlung Jahressumme bei {TARGET_TILT}° (interpoliert aus {available_tilts})")
print(f"F02: Wintereinstrahlung Okt–Mär bei {TARGET_TILT}° (interpoliert aus {available_tilts})")
print(f"Beide aus BFE-Kolumnen: {COL_JAHR} / {COL_WINTER}")

F01: Globalstrahlung Jahressumme bei 45° (interpoliert aus [0, 30, 75, 90])
F02: Wintereinstrahlung Okt–Mär bei 45° (interpoliert aus [0, 30, 75, 90])
Beide aus BFE-Kolumnen: Globalstrahlung_Jahressumme_kWhm2 / Globalstrahlung_Wintersumme_kWhm2


## 5. DEM-basierte Eignungsfaktoren

In [ ]:
print("=== F03: Hangneigung (Optimum 20–30°) ===")
f03 = normalize_piecewise(slope, [0, 20, 30, 45, 90], [0, 1, 1, 0, 0])
save_criterion(f03, "f03_hangneigung")

print("\n=== F04: Exposition (Südabweichung) ===")
south_dev = np.abs(aspect - 180)
f04 = normalize_linear(south_dev, 180, 0)
save_criterion(f04, "f04_exposition")

print("\n=== F05: Höhenlage (Optimum 1800–2500m) ===")
f05 = normalize_piecewise(dem, [1500, 1800, 2500, 2700], [0, 1, 1, 0])
save_criterion(f05, "f05_hoehenlage")

=== F03: Hangneigung (Optimum 20–30°) ===
  f03_hangneigung: 0.000–1.000 (mean 0.872)

=== F04: Exposition (Südabweichung) ===
  f04_exposition: 0.667–1.000 (mean 0.832)

=== F05: Höhenlage (Optimum 1800–2500m) ===
  f05_hoehenlage: 0.000–1.000 (mean 0.874)


## 6. SPASS Schneebedeckung

In [ ]:
print("=== F06: Schneebedeckung (SPASS) ===")
spass_files = sorted((RAW / "snow").glob("**/*.nc"))
if not spass_files:
    print("  ⚠ SPASS nicht gefunden — F06 übersprungen")
else:
    import xarray as xr
    ds = xr.open_dataset(spass_files[0], engine="h5netcdf", decode_times=False)
    swe_var = next((v for v in ds.data_vars if "swe" in v.lower()), list(ds.data_vars)[0])
    print(f"  Datei: {spass_files[0].name} | Var: {swe_var} | Dims: {dict(ds.dims)}")

    snow_frac = (ds[swe_var] > 0).mean(dim="time").values.astype(np.float32)

    if "E" in ds.dims:
        x, y = ds["E"].values, ds["N"].values
    elif "chx" in ds.dims:
        x, y = ds["chx"].values, ds["chy"].values
    else:
        x, y = ds[list(ds.dims)[1]].values, ds[list(ds.dims)[0]].values

    src_tf = from_bounds(x.min(), y.min(), x.max(), y.max(), len(x), len(y))
    snow_25m = np.full((height, width), NODATA, dtype=np.float32)
    reproject(source=np.flipud(snow_frac), destination=snow_25m,
              src_transform=src_tf, src_crs=CRS, dst_transform=transform, dst_crs=CRS,
              resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=NODATA)

    # F06: Schneebedeckung — Linear [0,1]: mehr Schneetage = höherer Albedo-Reflexionsbonus = besser
    #      Ref: Kahl et al. (2019) Mechanismus (ii) SCD, Frischholz et al. (2024)
    v_s = snow_25m[valid & (snow_25m != NODATA)]
    f06 = normalize_linear(snow_25m, v_s.min(), v_s.max())
    save_criterion(f06, "f06_schneebedeckung")
    print(f"  Daten: {v_s.min():.2f}–{v_s.max():.2f} | Mehr Schnee = besser")
    ds.close()

=== F06: Schneebedeckung (SPASS) ===
  Datei: SWECLQMD_ch01h.swiss.lv95_WY_1962_2023.nc | Var: SWECLQMD | Dims: {'time': 22645, 'E': 370, 'N': 265}
  f06_schneebedeckung: 0.000–1.000 (mean 0.727)


## 7. Distanz-Raster (Strasse, Netz, Infrastruktur)

In [ ]:
print("=== Distanz-Raster ===\n")

# Strassen aus swissTLM3D (nur befahrbare Strassen, ohne Wege/Pfade)
EXCLUDE_OBJEKTART = {"1m Weg", "2m Weg", "1m Wegfragment", "2m Wegfragment",
                     "Markierte Spur", "Klettersteig", "Faehre", "Autozug"}
strassen_all = gpd.read_file(str(tlm_path), layer="tlm_strassen_strasse", bbox=bbox)
strassen = strassen_all[~strassen_all["objektart"].isin(EXCLUDE_OBJEKTART)]
print(f"  Strassen: {len(strassen)} von {len(strassen_all)} (ohne Wege/Pfade)")
print(f"  Behaltene Typen: {strassen['objektart'].value_counts().to_dict()}")
del strassen_all
strassen_raster = rasterize_to_grid(strassen)
dist_strasse = compute_distance(strassen_raster, "dist_strasse_gr_25m")

# Stromnetz aus swissTLM3D (tlm_bauten_leitung — alle Netzebenen)
leitungen = gpd.read_file(str(tlm_path), layer="tlm_bauten_leitung", bbox=bbox)
print(f"  Stromleitungen: {len(leitungen)} Features (Netzebenen: {leitungen['netzebene'].value_counts().to_dict()})")
netz_raster = rasterize_to_grid(leitungen)
dist_netz = compute_distance(netz_raster, "dist_netz_gr_25m")

# Staumauern aus swissTLM3D
try:
    staumauern = gpd.read_file(str(tlm_path), layer="tlm_bauten_staubaute", bbox=bbox)
    print(f"  Staumauern: {len(staumauern)} Features")
    infra_raster = rasterize_to_grid(staumauern)
    dist_infra = compute_distance(infra_raster, "dist_infrastruktur_gr_25m")
except:
    print("  ⚠ Staumauer-Layer nicht gefunden")
    dist_infra = np.full((height, width), 10000, dtype=np.float32)

# Normalisieren (inverse linear [0,1]: kürzere Distanz = höherer Score)
for name, dist, fid in [("netzanschluss", dist_netz, "f07"),
                         ("strasse", dist_strasse, "f08"),
                         ("infrastruktur", dist_infra, "f09")]:
    v_d = dist[valid & (dist != NODATA)]
    f = normalize_linear(dist, v_d.min(), v_d.max(), invert=True)
    save_criterion(f, f"{fid}_{name}")
    print(f"  {fid}: {v_d.min():.0f}–{v_d.max():.0f} m (invers)")

=== Distanz-Raster ===

  Strassen: 187910 Features
  dist_strasse_gr_25m: max 2810 m
  Stromleitungen: 299 Features (Netzebenen: {'3': 132, '1': 128, 'ub': 39})
  dist_netz_gr_25m: max 17262 m
  Staumauern: 3057 Features
  dist_infrastruktur_gr_25m: max 11532 m
  f07_netzanschluss: 0.000–1.000 (mean 0.575)
  f08_strasse: 0.000–1.000 (mean 0.715)
  f09_infrastruktur: 0.000–1.000 (mean 0.509)


## 8. Sichtbarkeit (Proxy)

In [ ]:
print("=== F10: Sichtbarkeit (Distanz zu Siedlungen, swissTLM3D) ===")
gebaeude = gpd.read_file(str(tlm_path), layer="tlm_bauten_gebaeude_footprint", bbox=bbox)
print(f"  Gebäude-Footprints: {len(gebaeude)} Features")
siedl_raster = rasterize_to_grid(gebaeude)
dist_siedlung = distance_transform_edt((siedl_raster == 0).astype(float), sampling=(RES, RES)).astype(np.float32)
# F10: Inverse [0,1]: weniger sichtbar von Siedlungen = höherer Score
# Ref: Swissolar (2025) §4.2.1 — Sichtbarkeit und Raumwirkung
v_s = dist_siedlung[valid]
f10 = normalize_linear(dist_siedlung, v_s.min(), v_s.max())
save_criterion(f10, "f10_sichtbarkeit")
print(f"  Daten: {v_s.min():.0f}–{v_s.max():.0f} m | Weiter = weniger sichtbar = besser")

=== F10: Sichtbarkeit (Distanz zu Siedlungen, swissTLM3D) ===
  Gebäude-Footprints: 343695 Features
  f10_sichtbarkeit: 0.038–1.000 (mean 0.418)


## 9. Übersicht — Normalisierte Eignungsfaktoren

In [ ]:
# Alle 10 Kriterien laden und als Übersichtskarte darstellen
criteria_files = {
    "F01 Solar Radiation":       "f01_globalstrahlung",
    "F02 Winter Radiation":      "f02_wintereinstrahlung",
    "F03 Slope":                 "f03_hangneigung",
    "F04 Aspect":                "f04_exposition",
    "F05 Elevation":             "f05_hoehenlage",
    "F06 Snow Cover Duration":   "f06_schneebedeckung",
    "F07 Distance to Grid":      "f07_netzanschluss",
    "F08 Distance to Roads":     "f08_strasse",
    "F09 Distance to Infra":     "f09_infrastruktur",
    "F10 Visibility":            "f10_sichtbarkeit",
}

fig, axes = plt.subplots(2, 5, figsize=(24, 9))
fig.suptitle("Normalised Criteria Scores [0, 1]  –  Canton Graubünden", fontsize=15, y=0.98)

for ax, (label, fname) in zip(axes.flat, criteria_files.items()):
    path = PROC / f"criteria/{fname}.tif"
    if not path.exists():
        ax.set_title(f"{label}\n⚠ fehlt", fontsize=10)
        ax.set_axis_off()
        continue
    with rasterio.open(path) as src:
        data = src.read(1)
    display = np.where(valid & (data != NODATA), data, np.nan)
    im = ax.imshow(display, cmap="cividis", vmin=0, vmax=1)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_axis_off()
    plt.colorbar(im, ax=ax, label="Score [0–1]", shrink=0.75)

plt.tight_layout()
fig.savefig(OUT / "criteria_overview.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Gespeichert: {OUT / 'criteria_overview.png'}")

## Nächster Schritt

Alle Eignungsfaktoren liegen als normalisierte GeoTIFFs in `data/processed/criteria/` vor.

**→ Weiter mit:** WLC-Verschneidung (AHP-Gewichte × Kriterien × Constraint-Maske × Akzeptanz)

**Interface-Garantie:** Dieses Notebook benötigt aus 01/02a nur `constraint_mask_s2.tif` und die DEM-Ableitungen. Es kann mit einer beliebigen binären Maske betrieben werden.